# Làm sạch dữ liệu tin tức

Giữ bài phát hành từ 19/10/2021 đến 25/09/2026. Loại bài chỉ có QNS; bài nhiều mã chỉ bỏ QNS khỏi ticker.

Boilerplate được xử lý trước khi kiểm tra thân bài và gộp trùng. Bài chưa đủ chất lượng được lưu riêng, không tự bổ sung nội dung.

Kết quả gồm 8 trường trong `data/processed/news_clean.csv`; log nằm trong `data/interim`. Dữ liệu raw được giữ nguyên.

## 1. Chuẩn bị dữ liệu

In [1]:
import sys
from pathlib import Path

import pandas as pd

In [2]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / 'news_processing').exists():
    PROJECT_DIR = CURRENT_DIR / 'news_processing'
else:
    PROJECT_DIR = CURRENT_DIR

sys.path.insert(0, str(PROJECT_DIR))
RAW_FILE = PROJECT_DIR / 'data' / 'raw' / 'news_raw.csv'
VALIDATION_FILE = PROJECT_DIR / 'data' / 'interim' / 'validation_summary.csv'
INTERIM_DIR = PROJECT_DIR / 'data' / 'interim'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

START_DATE = pd.Timestamp('2021-10-19')
END_DATE = pd.Timestamp('2026-09-25')
EXCLUDED_TICKERS = {'QNS'}

In [3]:
from src.cleaning_utils import (
    assess_content_quality,
    apply_approved_content_corrections,
    clean_article_content,
    consolidate_article_duplicates,
    normalize_inline_text,
    normalize_tickers,
    normalize_timestamp,
    normalize_url,
    parse_serialized_list,
    resolve_published_date,
    residual_boilerplate_issues,
    sha256_text,
)
from src.validation_utils import (
    is_valid_http_url,
    normalize_title,
    split_tickers,
)

In [4]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f'Không tìm thấy dữ liệu raw: {RAW_FILE}')

raw_news = pd.read_csv(RAW_FILE, low_memory=False)
validation_summary = pd.read_csv(VALIDATION_FILE)
clean_news = raw_news.copy()
clean_news.insert(0, 'raw_row_index', clean_news.index)

print(f'Số bài raw: {len(raw_news):,}')
display(validation_summary)
display(raw_news.head(3))

Số bài raw: 4,491


,metric,value
0,eda_status,REQUIRES_CLEANING
1,total_rows,4491
2,total_columns,22
3,published_at_missing,858
4,date_conflicts,1116
5,serialized_list_rows,31
6,invalid_raw_urls,31
7,duplicate_canonical_url_rows,138
8,duplicate_title_candidates,559
9,duplicate_content_candidates,10


,news_id,source,url,title,summary,content,published_date,published_at_raw,author,category,...,content_hash,industry_group,tickers,keywords,event_type,note,dataset_origin,source_news_ids,source_content_hashes,original_crawl_status
0,3a140e495704f6a5369346873ed120b8c1befe72f62f65...,BNews,https://bnews.vn/masan-nhan-83-74-co-phan-tu-m...,"Masan nhận 83,74% cổ phần từ một công ty con c...","Masan sẽ nhận 83,47% cổ phần phổ thông đang lư...","BNEWS Masan sẽ nhận 83,47% cổ phần phổ thông đ...",2020-01-01,2020-01-01T20:08:35+07:00,Thành Trung/BNEWS/TTXVN,Chứng khoán,...,c8b15bca57b22cc143b2296b78de2f03bb09715cec695e...,NaN,MSN,NaN,NaN,NaN,main_crawler,3a140e495704f6a5369346873ed120b8c1befe72f62f65...,c8b15bca57b22cc143b2296b78de2f03bb09715cec695e...,success
1,0308045530c04aa5585bd6a12a26ee311465f7b79bdce9...,VietnamNet,https://vietnamnet.vn/nghe-vietnamnet-su-that-...,Nghe VietNamNet: Sự thật về 2.495 tỷ đồng SABE...,"Sau khi Bộ Công Thương, Bộ Tài chính có ý kiến...","Sau khi Bộ Công Thương, Bộ Tài chính có ý kiến...",2020-01-01,2020-01-01T11:17:11.000 +07:00,VietNamNet News,Nghe VietNamNet,...,734e0256a5d469c1ba28b520e573ba3c2909eac9be0516...,NaN,SAB,NaN,NaN,NaN,main_crawler,0308045530c04aa5585bd6a12a26ee311465f7b79bdce9...,734e0256a5d469c1ba28b520e573ba3c2909eac9be0516...,success
2,cb018360ff0b90d0232a6235cb705d22171969dac35eea...,VietnamNet,https://vietnamnet.vn/su-that-ve-2495-ty-dong-...,Sự thật về 2.495 tỷ đồng SABECO phải nộp vào n...,"Sau khi Bộ Công Thương, Bộ Tài chính có ý kiến...","Sau khi Bộ Công Thương, Bộ Tài chính có ý kiến...",2020-01-01,2020-01-01T10:35:58.000 +07:00,VietNamNet News,Tài chính,...,a8fa234274294748cba4343fa947cdbb27e57919258d2c...,NaN,SAB,NaN,NaN,NaN,main_crawler,cb018360ff0b90d0232a6235cb705d22171969dac35eea...,a8fa234274294748cba4343fa947cdbb27e57919258d2c...,success


## 2. Chuẩn hóa dữ liệu

### Trường văn bản chứa danh sách

Một số ô lưu dạng `['giá trị']` thay vì văn bản đơn. Cần nhận diện để chuẩn hóa đúng, không giữ nguyên dấu ngoặc và ký tự thừa.

In [5]:
SCALAR_COLUMNS = ['url', 'title', 'summary', 'author', 'category']

serialized_list_masks = {
    column: raw_news[column].map(parse_serialized_list).notna()
    for column in SCALAR_COLUMNS
}
serialized_list_report = pd.DataFrame([
    {'column': column, 'actual_serialized_lists': int(mask.sum())}
    for column, mask in serialized_list_masks.items()
])

actual_serialized_row_mask = pd.DataFrame(serialized_list_masks).any(axis=1)
display(serialized_list_report)
print('Số dòng chứa danh sách dạng chuỗi:', int(actual_serialized_row_mask.sum()))

,column,actual_serialized_lists
0,url,31
1,title,31
2,summary,31
3,author,25
4,category,25


Số dòng chứa danh sách dạng chuỗi: 31


### Văn bản và boilerplate

Menu, chân trang và tin liên quan được tách theo cấu trúc từng nguồn và tiêu đề đã xác minh. Không xóa theo từ chung như “đăng ký”, vì có thể mất thông tin tài chính.

In [6]:
INLINE_TEXT_COLUMNS = ['title', 'summary', 'author', 'category']
text_change_counts = {}

for column in INLINE_TEXT_COLUMNS:
    cleaned_values = raw_news[column].map(normalize_inline_text)
    original_text = raw_news[column].fillna('<NA>').astype(str)
    normalized_text = cleaned_values.fillna('<NA>').astype(str)
    text_change_counts[column] = int(original_text.ne(normalized_text).sum())
    clean_news[column] = cleaned_values

display(pd.Series(text_change_counts, name='rows_changed').to_frame())

,rows_changed
title,93
summary,184
author,37
category,25


In [7]:
KNOWN_ARTICLE_TITLES = tuple(clean_news['title'].dropna().unique())
cleaned_content = clean_news.apply(
    lambda row: clean_article_content(
        row['content'],
        source=row['source'],
        title=row['title'],
        summary=row['summary'],
        known_article_titles=KNOWN_ARTICLE_TITLES,
    ),
    axis=1,
)
content_change_count = int(
    raw_news['content'].fillna('<NA>').astype(str)
    .ne(cleaned_content.fillna('<NA>').astype(str))
    .sum()
)
clean_news['content'] = cleaned_content

print('Số bài được làm sạch nội dung:', f'{content_change_count:,}')

Số bài được làm sạch nội dung: 2,923


In [ ]:
clean_news, reviewed_content_log = apply_approved_content_corrections(clean_news)
print('Bài áp dụng sửa nội dung đã duyệt:', len(reviewed_content_log))

### URL bài báo

In [8]:
cleaned_urls = raw_news['url'].map(normalize_url)
url_change_count = int(
    raw_news['url'].fillna('<NA>').astype(str)
    .ne(cleaned_urls.fillna('<NA>').astype(str))
    .sum()
)
clean_news['url'] = cleaned_urls

invalid_clean_url_count = int((~clean_news['url'].map(is_valid_http_url)).sum())
print('URL được thay đổi:', f'{url_change_count:,}')
print('URL còn không hợp lệ:', invalid_clean_url_count)

URL được thay đổi: 137
URL còn không hợp lệ: 0


### Ngày phát hành

Ưu tiên thời điểm đăng từ nguồn khi đọc được. Giờ chỉ dùng để đối chiếu ngày theo múi giờ Việt Nam; bảng xuất giữ `published_date`, không thay bằng ngày crawl.

In [9]:
original_dates = pd.to_datetime(
    raw_news['published_date'], format='%Y-%m-%d', errors='coerce'
)
resolved_dates = pd.Series(
    [
        resolve_published_date(date, timestamp)
        for date, timestamp in zip(
            raw_news['published_date'], raw_news['published_at_raw']
        )
    ],
    index=raw_news.index,
)

clean_news['published_date'] = resolved_dates
clean_news['published_at'] = raw_news['published_at_raw'].map(normalize_timestamp)
clean_news['crawl_time'] = raw_news['crawl_time'].map(normalize_timestamp)

In [10]:
resolved_date_values = pd.to_datetime(resolved_dates, errors='coerce')
date_corrected_mask = (
    original_dates.notna()
    & resolved_date_values.notna()
    & original_dates.ne(resolved_date_values)
)

print('Ngày phát hành được sửa:', f'{int(date_corrected_mask.sum()):,}')
print('Số bài thiếu thời điểm đăng:', f"{int(clean_news['published_at'].isna().sum()):,}")
print('Ngày phát hành không đọc được:', int(resolved_date_values.isna().sum()))
display(clean_news.loc[date_corrected_mask, [
    'source', 'published_date', 'published_at_raw', 'published_at'
]].head(5))

Ngày phát hành được sửa: 1,116
Số bài thiếu thời điểm đăng: 858
Ngày phát hành không đọc được: 0


,source,published_date,published_at_raw,published_at
3,BNews,2020-04-01,2020-04-01T09:32:51+07:00,2020-04-01T09:32:51+07:00
4,VietnamNet,2020-04-01,2020-04-01T11:00:00.000 +07:00,2020-04-01T11:00:00+07:00
5,BNews,2020-07-01,2020-07-01T12:36:39+07:00,2020-07-01T12:36:39+07:00
6,BNews,2020-09-01,2020-09-01T19:46:27+07:00,2020-09-01T19:46:27+07:00
7,BNews,2020-10-01,2020-10-01T15:43:40+07:00,2020-10-01T15:43:40+07:00


### Mã cổ phiếu

In [11]:
original_ticker_lists = raw_news['tickers'].map(split_tickers)
qns_only_mask = original_ticker_lists.map(lambda values: set(values) == {'QNS'})
qns_mixed_mask = original_ticker_lists.map(
    lambda values: 'QNS' in values and len(set(values)) > 1
)

clean_news['tickers'] = raw_news['tickers'].map(
    lambda value: normalize_tickers(value, EXCLUDED_TICKERS)
)
ticker_change_count = int(
    raw_news['tickers'].astype(str).ne(clean_news['tickers']).sum()
)

print('Ticker được chuẩn hóa:', f'{ticker_change_count:,}')
print('Bài chỉ có QNS trong toàn bộ raw:', int(qns_only_mask.sum()))
print('Bài có QNS cùng mã khác:', int(qns_mixed_mask.sum()))

Ticker được chuẩn hóa: 218
Bài chỉ có QNS trong toàn bộ raw: 56
Bài có QNS cùng mã khác: 8


## 3. Lọc dữ liệu

### Khoảng thời gian

In [12]:
clean_dates = pd.to_datetime(clean_news['published_date'], errors='coerce')
date_scope_mask = clean_dates.between(START_DATE, END_DATE, inclusive='both')

date_excluded = clean_news.loc[~date_scope_mask].copy()
date_excluded['exclusion_reason'] = 'DATE_OUT_OF_SCOPE'
date_excluded['original_tickers'] = raw_news.loc[date_excluded.index, 'tickers']

scoped_news = clean_news.loc[date_scope_mask].copy()
print('Bản ghi trong khoảng ngày:', f'{len(scoped_news):,}')
print('Bản ghi ngoài khoảng ngày:', f'{len(date_excluded):,}')
print('Ngày nhỏ nhất thực tế:', scoped_news['published_date'].min())
print('Ngày lớn nhất thực tế:', scoped_news['published_date'].max())

Bản ghi trong khoảng ngày: 4,021
Bản ghi ngoài khoảng ngày: 470
Ngày nhỏ nhất thực tế: 2021-10-20
Ngày lớn nhất thực tế: 2026-09-25


### Bài chỉ có QNS

In [13]:
scoped_qns_only_mask = qns_only_mask.loc[scoped_news.index]
scoped_qns_mixed_mask = qns_mixed_mask.loc[scoped_news.index]

qns_excluded = scoped_news.loc[scoped_qns_only_mask].copy()
qns_excluded['exclusion_reason'] = 'QNS_ONLY'
qns_excluded['original_tickers'] = raw_news.loc[qns_excluded.index, 'tickers']

clean_news = scoped_news.loc[~scoped_qns_only_mask].copy()
print('Bài chỉ có QNS bị loại:', len(qns_excluded))
print('Bài nhiều mã được giữ và bỏ QNS:', int(scoped_qns_mixed_mask.sum()))
print('Số bài còn lại trước khi kiểm tra thân bài:', f'{len(clean_news):,}')

Bài chỉ có QNS bị loại: 53
Bài nhiều mã được giữ và bỏ QNS: 8
Số bài còn lại trước khi kiểm tra thân bài: 3,968


### Chất lượng thân bài

Kết hợp nguồn, tiêu đề, tóm tắt và nội dung để phát hiện thân bài thiếu hoặc sai. Bài dưới 200 ký tự hoặc chưa rõ ranh giới nội dung được lưu riêng; không dùng tóm tắt để dựng lại bài.

In [14]:
body_issues = clean_news.apply(
    lambda row: assess_content_quality(
        row['content'],
        source=row['source'],
        title=row['title'],
        summary=row['summary'],
        url=row['url'],
    ),
    axis=1,
)
short_body_mask = clean_news['content'].fillna('').str.len().lt(200)
for row_index in body_issues.index[short_body_mask]:
    body_issues.at[row_index] = sorted(
        set(body_issues.at[row_index] + ['CONTENT_TOO_SHORT'])
    )

In [15]:
body_excluded_mask = body_issues.map(bool)
body_excluded = clean_news.loc[body_excluded_mask].copy()
body_excluded['exclusion_reason'] = body_issues.loc[body_excluded_mask].map(
    lambda issues: 'CONTENT_QUALITY:' + '|'.join(issues)
)
body_excluded['original_tickers'] = raw_news.loc[body_excluded.index, 'tickers']
clean_news = clean_news.loc[~body_excluded_mask].copy()

print('Số bài chưa đủ chất lượng nội dung:', len(body_excluded))
display(body_excluded['exclusion_reason'].value_counts().rename('rows').to_frame())

Số bài chưa đủ chất lượng nội dung: 336


,rows
exclusion_reason,
CONTENT_QUALITY:UNRESOLVED_TRAILING_FRAGMENT,259
CONTENT_QUALITY:RELATED_LINKS_WITHOUT_BODY,45
CONTENT_QUALITY:CONTENT_TOO_SHORT|SUMMARY_ONLY,22
CONTENT_QUALITY:MISMATCHED_BODY,3
CONTENT_QUALITY:CONTENT_TOO_SHORT|INFOGRAPHIC_WITHOUT_TEXT|TITLE_ONLY,3
CONTENT_QUALITY:CATEGORY_PAGE,2
CONTENT_QUALITY:PODCAST_WITHOUT_TRANSCRIPT,1
CONTENT_QUALITY:SUMMARY_ONLY,1


## 4. Gộp bài trùng

Gom các bản ghi trùng URL, nội dung hoặc tiêu đề chuẩn hóa. Ưu tiên bản có nhiều thông tin, có thời điểm đăng từ nguồn, rồi xét độ dài nội dung. Gộp ticker và lưu ID gốc để đối chiếu.

In [16]:
rows_before_deduplication = len(clean_news)
clean_news, duplicate_merge_log = consolidate_article_duplicates(clean_news)
duplicate_rows_removed = rows_before_deduplication - len(clean_news)

print('Số bài trước khi gộp trùng:', f'{rows_before_deduplication:,}')
print('Nhóm trùng được hợp nhất:', len(duplicate_merge_log))
print('Bản ghi trùng bị loại:', duplicate_rows_removed)
print('Bản ghi còn lại:', f'{len(clean_news):,}')
display(duplicate_merge_log['duplicate_reasons'].value_counts().to_frame('groups'))

Số bài trước khi gộp trùng: 3,632
Nhóm trùng được hợp nhất: 257
Bản ghi trùng bị loại: 276
Bản ghi còn lại: 3,356


,groups
duplicate_reasons,
normalized_title,191
canonical_url|normalized_title,64
exact_content,1
exact_content|normalized_title,1


## 5. Chuẩn bị bảng kết quả

In [17]:
clean_news['content_hash'] = clean_news['content'].map(sha256_text)
clean_news = clean_news.sort_values(
    ['published_date', 'source', 'title', 'news_id'],
    kind='stable',
).reset_index(drop=True)

raw_columns = list(raw_news.columns)
published_at_position = raw_columns.index('published_at_raw') + 2
working_columns = ['raw_row_index'] + raw_columns
working_columns.insert(published_at_position, 'published_at')
clean_news = clean_news[working_columns]

In [18]:
OUTPUT_COLUMNS = [
    'news_id', 'source', 'url', 'title',
    'summary', 'content', 'published_date', 'tickers',
]
export_news = clean_news[OUTPUT_COLUMNS].copy()

print('Kích thước bảng kết quả:', export_news.shape)
display(export_news.head(3))

Kích thước bảng kết quả: (3356, 8)


,news_id,source,url,title,summary,content,published_date,tickers
0,57a3bac8e0b55bd495bf688a17e6a9f3a3b6fef06373ae...,BNews,https://bnews.vn/masan-lien-tiep-khai-truong-c...,"Masan liên tiếp khai trương cửa hàng CVLife, t...",Sau khi công bố lấn sân sang lĩnh vực viễn thô...,Sau khi công bố lấn sân sang lĩnh vực viễn thô...,2021-10-20,MSN
1,9cddfe019c634b81361fb00a77a9bae5bbc61795e00ce8...,BNews,https://bnews.vn/lien-doanh-del-monte-vinamilk...,Liên doanh Del Monte-Vinamilk tại Philippines ...,Công ty liên doanh Del Monte-Vinamilk mới ra m...,Công ty liên doanh Del Monte-Vinamilk mới ra m...,2021-10-21,VNM
2,e1751ac1f9ba5c069ce22bae353e052684d1f59c8d23b3...,BNews,https://bnews.vn/vinamilk-ung-ho-10-ty-dong-va...,Vinamilk ủng hộ 10 tỷ đồng và 1 triệu ly sữa c...,"Tại Bộ Lao động - Thương binh và Xã hội, Vinam...","Tại Bộ Lao động - Thương binh và Xã hội, Vinam...",2021-10-22,VNM


## 6. Kiểm tra dữ liệu sau làm sạch

Đối chiếu nội dung, ngày, ticker, trùng lặp và 8 cột đầu ra. Kiểm tra boilerplate chỉ bao phủ các mẫu đã nhận diện, không phát hiện được mọi nội dung thừa.

In [19]:
REQUIRED_COLUMNS = [
    'news_id', 'source', 'url', 'title', 'content', 'published_date',
    'crawl_time', 'content_hash', 'tickers', 'dataset_origin',
]
ALLOWED_TICKERS = {
    'ANV', 'DBC', 'DGW', 'DHG', 'FRT', 'IMP', 'MSN',
    'MWG', 'PNJ', 'SAB', 'SBT', 'VHC', 'VNM',
}

required_empty_mask = clean_news[REQUIRED_COLUMNS].apply(
    lambda column: column.fillna('').astype(str).str.strip().eq('')
).any(axis=1)
clean_date_values = pd.to_datetime(clean_news['published_date'], errors='coerce')
clean_ticker_lists = clean_news['tickers'].map(split_tickers)
unknown_ticker_mask = clean_ticker_lists.map(
    lambda values: bool(set(values) - ALLOWED_TICKERS)
)

In [20]:
published_at_values = pd.to_datetime(
    clean_news['published_at'], errors='coerce', utc=True
).dt.tz_convert('Asia/Ho_Chi_Minh')
published_at_dates = published_at_values.dt.tz_localize(None).dt.normalize()
date_conflict_mask = (
    published_at_dates.notna()
    & clean_date_values.ne(published_at_dates)
)
serialized_list_remaining_mask = clean_news[SCALAR_COLUMNS].apply(
    lambda column: column.map(parse_serialized_list).notna()
).any(axis=1)

In [21]:
remaining_boilerplate = clean_news.apply(
    lambda row: bool(
        residual_boilerplate_issues(row['content'], source=row['source'])
    ),
    axis=1,
)
remaining_body_issues = clean_news.apply(
    lambda row: bool(assess_content_quality(
        row['content'], source=row['source'], title=row['title'],
        summary=row['summary'], url=row['url'],
    )),
    axis=1,
)
hash_mismatch_mask = clean_news['content_hash'].ne(
    clean_news['content'].map(sha256_text)
)
normalized_clean_titles = clean_news['title'].map(normalize_title)
duplicate_title_mask = normalized_clean_titles.duplicated(keep=False)
duplicate_content_mask = clean_news['content'].duplicated(keep=False)
duplicate_news_id_mask = clean_news['news_id'].duplicated(keep=False)

In [22]:
invalid_url_count = int((~clean_news['url'].map(is_valid_http_url)).sum())
out_of_scope_count = int(
    (~clean_date_values.between(START_DATE, END_DATE, inclusive='both')).sum()
)
remaining_qns_count = int(
    clean_ticker_lists.map(lambda values: 'QNS' in values).sum()
)
short_content_count = int(clean_news['content'].str.len().lt(200).sum())
row_balance = abs(
    len(raw_news) - len(export_news) - len(date_excluded)
    - len(qns_excluded) - len(body_excluded) - duplicate_rows_removed
)

In [23]:
quality_check_counts = [
    ('Còn mẫu boilerplate đã nhận diện', int(remaining_boilerplate.sum())),
    ('Thân bài không đạt chất lượng', int(remaining_body_issues.sum())),
    ('Trường bắt buộc rỗng', int(required_empty_mask.sum())),
    ('URL không hợp lệ', invalid_url_count),
    ('Ngày ngoài phạm vi', out_of_scope_count),
    ('Ngày không khớp published_at', int(date_conflict_mask.sum())),
    ('Còn token QNS', remaining_qns_count),
    ('Ticker ngoài danh mục', int(unknown_ticker_mask.sum())),
    ('Còn list serialize', int(serialized_list_remaining_mask.sum())),
    ('Content dưới 200 ký tự', short_content_count),
    ('URL còn trùng', int(clean_news['url'].duplicated(keep=False).sum())),
    ('news_id còn trùng', int(duplicate_news_id_mask.sum())),
    ('Title chuẩn hóa còn trùng', int(duplicate_title_mask.sum())),
    ('Content còn trùng', int(duplicate_content_mask.sum())),
    ('Content hash không khớp', int(hash_mismatch_mask.sum())),
    ('Schema output không đúng 8 cột', int(list(export_news.columns) != OUTPUT_COLUMNS)),
    ('Đối soát số dòng raw không khớp', row_balance),
]
quality_checks = pd.DataFrame(
    quality_check_counts, columns=['check', 'issue_count']
)
clean_validation_status = (
    'PASS' if quality_checks['issue_count'].eq(0).all() else 'FAIL'
)

In [24]:
display(quality_checks)
print('Kết quả kiểm tra sau làm sạch:', clean_validation_status)
assert clean_validation_status == 'PASS', 'Dữ liệu clean chưa đạt điều kiện bắt buộc.'

,check,issue_count
0,Còn mẫu boilerplate đã nhận diện,0
1,Thân bài không đạt chất lượng,0
2,Trường bắt buộc rỗng,0
3,URL không hợp lệ,0
4,Ngày ngoài phạm vi,0
5,Ngày không khớp published_at,0
6,Còn token QNS,0
7,Ticker ngoài danh mục,0
8,Còn list serialize,0
9,Content dưới 200 ký tự,0


Kết quả kiểm tra sau làm sạch: PASS


### Các bản ghi còn trùng

In [25]:
duplicate_review_mask = (
    clean_news['url'].duplicated(keep=False)
    | duplicate_title_mask
    | duplicate_content_mask
)
duplicate_candidates_clean = clean_news.loc[duplicate_review_mask, [
    'news_id', 'source', 'url', 'title', 'published_date',
    'tickers', 'content_hash',
]].copy()
duplicate_candidates_clean.insert(
    0, 'normalized_title', normalized_clean_titles[duplicate_review_mask].to_numpy()
)
print('Bản ghi trùng còn lại:', len(duplicate_candidates_clean))
display(duplicate_candidates_clean.head(10))

Bản ghi trùng còn lại: 0


,normalized_title,news_id,source,url,title,published_date,tickers,content_hash


## 7. Lịch sử xử lý

In [26]:
cleaning_steps = [
    ('01', 'Parse serialized scalar lists', int(actual_serialized_row_mask.sum()), 'KEEP_AND_FIX'),
    ('02', 'Clean source-specific article content', content_change_count, 'KEEP_AND_FIX'),
    ('03', 'Canonicalize URL', url_change_count, 'KEEP_AND_FIX'),
    ('04', 'Correct published_date from source timestamp', int(date_corrected_mask.sum()), 'KEEP_AND_FIX'),
    ('05', 'Normalize ticker strings', ticker_change_count, 'KEEP_AND_FIX'),
    ('06', 'Remove records outside date scope', len(date_excluded), 'EXCLUDE'),
    ('07', 'Remove QNS-only records', len(qns_excluded), 'EXCLUDE'),
    ('08', 'Remove QNS token from mixed-ticker records', int(scoped_qns_mixed_mask.sum()), 'KEEP_AND_FIX'),
    ('09', 'Exclude unusable article bodies', len(body_excluded), 'EXCLUDE'),
    ('10', 'Merge duplicate articles', duplicate_rows_removed, 'MERGE'),
]
cleaning_log = pd.DataFrame(
    cleaning_steps, columns=['step', 'action', 'rows_affected', 'decision']
)

display(cleaning_log)

,step,action,rows_affected,decision
0,01,Parse serialized scalar lists,31,KEEP_AND_FIX
1,02,Clean source-specific article content,2923,KEEP_AND_FIX
2,03,Canonicalize URL,137,KEEP_AND_FIX
3,04,Correct published_date from source timestamp,1116,KEEP_AND_FIX
4,05,Normalize ticker strings,218,KEEP_AND_FIX
5,06,Remove records outside date scope,470,EXCLUDE
6,07,Remove QNS-only records,53,EXCLUDE
7,08,Remove QNS token from mixed-ticker records,8,KEEP_AND_FIX
8,09,Exclude unusable article bodies,336,EXCLUDE
9,10,Merge duplicate articles,276,MERGE


In [27]:
EXCLUSION_COLUMNS = [
    'raw_row_index', 'news_id', 'source', 'url', 'title',
    'published_date', 'original_tickers', 'exclusion_reason', 'summary', 'content',
]
excluded_records = pd.concat(
    [date_excluded, qns_excluded, body_excluded], ignore_index=True
)
excluded_records = excluded_records[EXCLUSION_COLUMNS]

display(excluded_records['exclusion_reason'].value_counts().to_frame('rows'))

,rows
exclusion_reason,
DATE_OUT_OF_SCOPE,470
CONTENT_QUALITY:UNRESOLVED_TRAILING_FRAGMENT,259
QNS_ONLY,53
CONTENT_QUALITY:RELATED_LINKS_WITHOUT_BODY,45
CONTENT_QUALITY:CONTENT_TOO_SHORT|SUMMARY_ONLY,22
CONTENT_QUALITY:MISMATCHED_BODY,3
CONTENT_QUALITY:CONTENT_TOO_SHORT|INFOGRAPHIC_WITHOUT_TEXT|TITLE_ONLY,3
CONTENT_QUALITY:CATEGORY_PAGE,2
CONTENT_QUALITY:PODCAST_WITHOUT_TRANSCRIPT,1


## 8. Lưu kết quả

In [28]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_tables = {
    PROCESSED_DIR / 'news_clean.csv': export_news,
    INTERIM_DIR / 'news_cleaning_log.csv': cleaning_log,
    INTERIM_DIR / 'news_excluded_records.csv': excluded_records,
    INTERIM_DIR / 'duplicate_merge_log.csv': duplicate_merge_log,
    INTERIM_DIR / 'clean_validation_summary.csv': quality_checks,
}

for output_path, output_table in output_tables.items():
    output_table.to_csv(output_path, index=False, encoding='utf-8-sig')

output_report = pd.DataFrame([
    {'file': path.relative_to(PROJECT_DIR).as_posix(), 'rows': len(table)}
    for path, table in output_tables.items()
])
display(output_report)

,file,rows
0,data/processed/news_clean.csv,3356
1,data/interim/news_cleaning_log.csv,10
2,data/interim/news_excluded_records.csv,859
3,data/interim/duplicate_merge_log.csv,257
4,data/interim/clean_validation_summary.csv,17
